In [ ]:
from os import cpu_count
from time import perf_counter

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc_extras as pmx
from joblib import Parallel, delayed
from sklearn.datasets import fetch_covtype
from pymc_extras.inference import estimate_parametric, fit_consensus_mc, merge_consensus, merge_parametric


# Consensus and Parametric MC

This notebook demonstrates the high-level Consensus Monte Carlo interface, the parametric Monte Carlo variant, and the standalone merge utilities in `pymc_extras.inference`.

The running example is a nonlinear regression with 2,000 IID observations. The mean function combines polynomial terms and multiple Fourier harmonics, giving a visibly curved response surface with enough data for sharding to matter while keeping the posterior continuous and unconstrained for Euclidean merging.


## Simulate a nonlinear regression data set

The predictor is one-dimensional, but the model uses nonlinear basis functions: an intercept, linear and quadratic terms, and three sine/cosine harmonics. The data-generating coefficients are known, and the observation scale is fixed in the model so `beta` is the only free random variable reconstructed by the MC mergers.


In [ ]:
RANDOM_SEED = 8927
N = 2_000
NOISE_SD = 0.30
X_MIN = -np.pi
X_MAX = np.pi
COEFFICIENTS = [
    'Intercept',
    'x',
    'x^2',
    'sin(x)',
    'cos(x)',
    'sin(2x)',
    'cos(2x)',
    'sin(3x)',
    'cos(3x)',
]
TRUE_BETA = np.array([0.25, 0.35, -0.08, 1.00, -0.35, 0.55, 0.25, -0.40, 0.15])


def nonlinear_design(x):
    x = np.asarray(x)
    return np.column_stack(
        [
            np.ones(x.shape[0]),
            x,
            x**2,
            np.sin(x),
            np.cos(x),
            np.sin(2 * x),
            np.cos(2 * x),
            np.sin(3 * x),
            np.cos(3 * x),
        ]
    )


rng = np.random.default_rng(RANDOM_SEED)
x = rng.uniform(X_MIN, X_MAX, size=N)
X = nonlinear_design(x)
mean_true = X @ TRUE_BETA
y = rng.normal(mean_true, NOISE_SD)

train_prop = 0.8
indices = rng.permutation(N)
n_train = int(train_prop * N)
train_idx = indices[:n_train]
test_idx = indices[n_train:]

x_train = x[train_idx]
X_train = X[train_idx]
y_train = y[train_idx]
x_test = x[test_idx]
X_test = X[test_idx]
y_test = y[test_idx]

pd.Series(
    {
        'n_train': X_train.shape[0],
        'n_test': X_test.shape[0],
        'n_basis': X_train.shape[1],
        'noise_sd': NOISE_SD,
    },
    name='data_summary',
)


## Model definition

`fit_consensus_mc` can split mutable data containers by named dimensions. The model names the observation dimension `obs_id` and the basis dimension `basis`. The only continuous free random variable is the global coefficient vector `beta`; the fixed observation scale keeps this example focused on posterior combination rather than constrained-parameter merging.


In [ ]:
def build_nonlinear_model(X_train, y_train, labels) -> pm.Model:
    coords = {'obs_id': np.arange(X_train.shape[0]), 'basis': labels}
    with pm.Model(coords=coords) as model:
        X_data = pm.Data('X', X_train, dims=('obs_id', 'basis'))
        y_data = pm.Data('y', y_train, dims='obs_id')
        beta = pm.Normal('beta', mu=0, sigma=1.5, dims='basis')
        mu = pm.math.dot(X_data, beta)
        pm.Normal('obs', mu=mu, sigma=NOISE_SD, observed=y_data, dims='obs_id')
    return model


## Helper functions

The merged consensus and parametric outputs reconstruct only free random variables. Predictions are therefore computed directly from posterior `beta` draws instead of relying on a deterministic mean node inside the model.


In [ ]:
def posterior_dataset(idata):
    posterior = idata['posterior']
    return posterior.dataset if hasattr(posterior, 'dataset') else idata.posterior


def beta_draws(idata):
    beta = posterior_dataset(idata)['beta']
    return beta.stack(sample=('chain', 'draw')).transpose('sample', 'basis').to_numpy()


def coefficient_stats(idata, fit_name):
    draws = beta_draws(idata)
    quantiles = np.quantile(draws, [0.03, 0.5, 0.97], axis=0)
    return pd.DataFrame(
        {
            'fit': fit_name,
            'coefficient': COEFFICIENTS,
            'true': TRUE_BETA,
            'mean': draws.mean(axis=0),
            'sd': draws.std(axis=0, ddof=1),
            'q03': quantiles[0],
            'q50': quantiles[1],
            'q97': quantiles[2],
        }
    )


def predict_mean(idata, X):
    return X @ beta_draws(idata).T


def posterior_mean_prediction(idata, X):
    return predict_mean(idata, X).mean(axis=1)


def rmse(idata, X, y):
    residual = posterior_mean_prediction(idata, X) - y
    return np.sqrt(np.mean(residual**2))


def mc_metadata(idata):
    data = idata['consensus_mc'].dataset
    return pd.Series(
        {
            'merge_method': data['merge_method'].item(),
            'diagonal': data['diagonal'].item(),
            'num_shards': data['num_shards'].item(),
            'draws': data['draws'].item(),
            'sample_draws': data['sample_draws'].item(),
            'tune': data['tune'].item(),
            'chains': data['chains'].item(),
            'cores': data['cores'].item(),
            'prior_scale': data['prior_scale'].item(),
            'shard_size': data['shard_size'].to_numpy().tolist(),
        }
    )


## Full-data reference fit

The reference posterior fits the full training set without sharding. It is not a gold standard for the approximations; it is a direct MCMC baseline for this simulated nonlinear regression and model specification.


In [ ]:
DRAWS = 300
TUNE = 300
SAMPLE_DRAWS = 300
NUM_SHARDS = 4

nonlinear_model = build_nonlinear_model(X_train, y_train, COEFFICIENTS)

with nonlinear_model:
    reference_idata = pm.sample(
        draws=DRAWS,
        tune=TUNE,
        chains=2,
        cores=1,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
        progressbar=False,
    )

print(az.summary(reference_idata, var_names=['beta'])[['mean', 'sd', 'ess_bulk', 'ess_tail', 'r_hat']])


## High-level Consensus MC with automatic splitting

`split_data={'X': 'obs_id', 'y': 'obs_id'}` shards both the nonlinear design matrix and the observed response along the named observation dimension. The metadata records `prior_scale = 1 / NUM_SHARDS`, because each shard model receives a fractional prior contribution before the shard posterior draws are merged.


In [ ]:
base_mc_kwargs = dict(
    model=nonlinear_model,
    split_data={'X': 'obs_id', 'y': 'obs_id'},
    num_shards=NUM_SHARDS,
    draws=DRAWS,
    sample_draws=SAMPLE_DRAWS,
    tune=TUNE,
    chains=1,
    cores=1,
    diagonal=False,
    sample_kwargs={'target_accept': 0.9},
    attach_data=True,
    progressbar=False,
)
consensus_idata = fit_consensus_mc(
    **base_mc_kwargs,
    random_seed=RANDOM_SEED + 1,
)

print(mc_metadata(consensus_idata).to_string())


## Parametric MC through `pmx.fit`

The dispatcher path uses the same model and automatic split configuration. With `merge_method='parametric'`, the shard posteriors define a Gaussian product approximation and the merger draws `DRAWS` samples from that approximation.


In [ ]:
parametric_idata = pmx.fit(
    method='consensus_mc',
    **base_mc_kwargs,
    merge_method='parametric',
    random_seed=RANDOM_SEED + 2,
)

print(mc_metadata(parametric_idata).to_string())


## Explicit shards and the diagonal approximation

Automatic splitting is convenient when all shard-local data are mutable model data containers. Explicit shards are useful when a workflow already has shard arrays. Non-scalar shard arrays need `sharded_dims=['obs_id']` or shard coordinates so the implementation can distinguish shard-local observation dimensions from global parameter dimensions.


In [ ]:
shard_indices = np.array_split(np.arange(X_train.shape[0]), NUM_SHARDS)
explicit_shards = [
    {'X': X_train[idx], 'y': y_train[idx]}
    for idx in shard_indices
]
explicit_shard_coords = [{'obs_id': idx} for idx in shard_indices]

explicit_diagonal_idata = fit_consensus_mc(
    model=nonlinear_model,
    shards=explicit_shards,
    shard_coords=explicit_shard_coords,
    sharded_dims=['obs_id'],
    draws=DRAWS,
    sample_draws=SAMPLE_DRAWS,
    tune=TUNE,
    chains=1,
    cores=1,
    diagonal=True,
    random_seed=RANDOM_SEED + 3,
    sample_kwargs={'target_accept': 0.9},
    attach_data=True,
    progressbar=False,
)

print(mc_metadata(explicit_diagonal_idata).to_string())


## Compare fitted posteriors and nonlinear predictions

The merged posterior DataTrees have a single reconstructed chain, so `r_hat` on merged outputs is not a convergence diagnostic. Use the full-data reference diagnostics, coefficient mean differences, and held-out RMSE deltas as sanity checks for the approximation in this notebook.


In [ ]:
fitted = {
    'reference': reference_idata,
    'consensus': consensus_idata,
    'parametric': parametric_idata,
    'explicit_diagonal': explicit_diagonal_idata,
}
coefficient_summary = pd.concat(
    [coefficient_stats(idata, name) for name, idata in fitted.items()],
    ignore_index=True,
)
rmse_table = pd.DataFrame.from_dict(
    {name: rmse(idata, X_test, y_test) for name, idata in fitted.items()},
    orient='index',
    columns=['rmse'],
)
mean_table = coefficient_summary.pivot(index='coefficient', columns='fit', values='mean')
approx_mean_error = mean_table[['consensus', 'parametric', 'explicit_diagonal']].sub(
    mean_table['reference'], axis=0
).abs()
true_mean_error = mean_table.sub(pd.Series(TRUE_BETA, index=COEFFICIENTS), axis=0).abs()
rmse_delta = rmse_table['rmse'].sub(rmse_table.loc['reference', 'rmse']).abs()

print(coefficient_summary.to_string(index=False))
print(rmse_table.to_string())
print(approx_mean_error.to_string())
print(true_mean_error.to_string())
print(rmse_delta.to_string())


In [ ]:
fit_order = ['reference', 'consensus', 'parametric', 'explicit_diagonal']
colors = dict(zip(fit_order, plt.rcParams['axes.prop_cycle'].by_key()['color'][: len(fit_order)]))
y_positions = np.arange(len(COEFFICIENTS))

fig, ax = plt.subplots(figsize=(9, 6))
for offset, fit_name in zip(np.linspace(-0.24, 0.24, len(fit_order)), fit_order):
    stats = coefficient_summary[coefficient_summary['fit'] == fit_name].set_index('coefficient').loc[COEFFICIENTS]
    positions = y_positions + offset
    lower = stats['mean'] - stats['q03']
    upper = stats['q97'] - stats['mean']
    ax.errorbar(
        stats['mean'],
        positions,
        xerr=np.vstack([lower, upper]),
        fmt='o',
        capsize=3,
        label=fit_name,
        color=colors[fit_name],
    )

ax.scatter(TRUE_BETA, y_positions, marker='x', s=80, color='black', label='true beta')
ax.axvline(0, color='0.8', linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(COEFFICIENTS)
ax.set_xlabel('Posterior coefficient mean and 3%–97% interval')
ax.set_title('Nonlinear basis coefficient summaries by fit')
ax.legend()
fig.tight_layout()


In [ ]:
x_grid = np.linspace(X_MIN, X_MAX, 300)
X_grid = nonlinear_design(x_grid)
curve_table = pd.DataFrame(
    {
        'x': x_grid,
        'true_mean': X_grid @ TRUE_BETA,
        'reference': posterior_mean_prediction(reference_idata, X_grid),
        'consensus': posterior_mean_prediction(consensus_idata, X_grid),
        'parametric': posterior_mean_prediction(parametric_idata, X_grid),
    }
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x_test, y_test, color='0.65', s=12, alpha=0.45, label='held-out observations')
ax.plot(curve_table['x'], curve_table['true_mean'], color='black', linewidth=2, label='true mean')
for fit_name in ['reference', 'consensus', 'parametric']:
    ax.plot(curve_table['x'], curve_table[fit_name], label=fit_name)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Nonlinear held-out prediction curves')
ax.legend()
fig.tight_layout()


## Real-world speed example: Forest CoverType classification

Consensus MC is most useful when the likelihood factorizes over many observations and full-data MCMC spends most of its time evaluating that likelihood. The Forest CoverType data loaded by `sklearn.datasets.fetch_covtype` has 581,012 rows and 54 predictors; this example uses the first ten feature columns and a binary target, `cover type == 2`, to keep the posterior low-dimensional while retaining the large observation count.

The high-level `fit_consensus_mc` helper currently samples shards sequentially, so this timing example fits shard models in parallel with `joblib` and merges the flattened coefficient draws with the public `merge_consensus` utility. The speed table compares the observed full-data NUTS wall time to the observed parallel shard wall time plus merge time.


In [ ]:
COVTYPE_RANDOM_SEED = 9427
COVTYPE_DRAWS = 200
COVTYPE_TUNE = 200
COVTYPE_SAMPLE_DRAWS = 200
COVTYPE_NUM_SHARDS = 8
COVTYPE_N_JOBS = min(COVTYPE_NUM_SHARDS, cpu_count() or 1)
COVTYPE_FEATURE_COUNT = 10
COVTYPE_MAX_ROWS = None
COVTYPE_TEST_ROWS = 20_000

covtype = fetch_covtype()
covtype_X_raw = covtype.data[:, :COVTYPE_FEATURE_COUNT].astype('float64', copy=False)
covtype_y_raw = (covtype.target == 2).astype('int8', copy=False)
covtype_feature_names = np.asarray(covtype.feature_names[:COVTYPE_FEATURE_COUNT])

if COVTYPE_MAX_ROWS is not None:
    covtype_row_rng = np.random.default_rng(COVTYPE_RANDOM_SEED)
    covtype_rows = covtype_row_rng.choice(
        covtype_X_raw.shape[0], size=COVTYPE_MAX_ROWS, replace=False
    )
    covtype_X_raw = covtype_X_raw[covtype_rows]
    covtype_y_raw = covtype_y_raw[covtype_rows]

covtype_rng = np.random.default_rng(COVTYPE_RANDOM_SEED)
covtype_indices = covtype_rng.permutation(covtype_X_raw.shape[0])
covtype_n_train = int(0.8 * covtype_X_raw.shape[0])
covtype_train_idx = covtype_indices[:covtype_n_train]
covtype_test_idx = covtype_indices[covtype_n_train:]
covtype_eval_idx = covtype_test_idx[: min(COVTYPE_TEST_ROWS, covtype_test_idx.size)]

covtype_mean = covtype_X_raw[covtype_train_idx].mean(axis=0)
covtype_scale = covtype_X_raw[covtype_train_idx].std(axis=0)
covtype_X = (covtype_X_raw - covtype_mean) / covtype_scale

covtype_X_train = covtype_X[covtype_train_idx]
covtype_y_train = covtype_y_raw[covtype_train_idx]
covtype_X_eval = covtype_X[covtype_eval_idx]
covtype_y_eval = covtype_y_raw[covtype_eval_idx]

covtype_data_summary = pd.Series(
    {
        'rows_total': covtype_X_raw.shape[0],
        'rows_train': covtype_X_train.shape[0],
        'rows_eval': covtype_X_eval.shape[0],
        'features_used': COVTYPE_FEATURE_COUNT,
        'positive_rate_train': covtype_y_train.mean(),
        'num_shards': COVTYPE_NUM_SHARDS,
        'parallel_jobs': COVTYPE_N_JOBS,
    },
    name='covtype_data_summary',
)
print(covtype_data_summary.to_string())


In [ ]:
def build_covtype_model(X_train, y_train, labels) -> pm.Model:
    coords = {'covtype_obs': np.arange(X_train.shape[0]), 'covtype_feature': labels}
    with pm.Model(coords=coords) as model:
        X_data = pm.Data('covtype_X', X_train, dims=('covtype_obs', 'covtype_feature'))
        y_data = pm.Data('covtype_y', y_train, dims='covtype_obs')
        intercept = pm.Normal('cov_intercept', mu=0, sigma=2.5)
        beta = pm.Normal('cov_beta', mu=0, sigma=1.0, dims='covtype_feature')
        logit_p = intercept + pm.math.dot(X_data, beta)
        pm.Bernoulli('cov_obs', logit_p=logit_p, observed=y_data, dims='covtype_obs')
    return model


def covtype_flat_draws(idata):
    posterior = posterior_dataset(idata)
    intercept = posterior['cov_intercept'].stack(sample=('chain', 'draw')).to_numpy()
    beta = (
        posterior['cov_beta']
        .stack(sample=('chain', 'draw'))
        .transpose('sample', 'covtype_feature')
        .to_numpy()
    )
    return np.column_stack([intercept, beta])


def covtype_probabilities(flat_draws, X):
    logits = flat_draws[:, :1] + flat_draws[:, 1:] @ X.T
    logits = np.clip(logits, -30, 30)
    return (1.0 / (1.0 + np.exp(-logits))).mean(axis=0)


def covtype_binary_metrics(flat_draws, X, y):
    probability = covtype_probabilities(flat_draws, X)
    probability = np.clip(probability, np.finfo(float).eps, 1 - np.finfo(float).eps)
    return pd.Series(
        {
            'log_loss': -(y * np.log(probability) + (1 - y) * np.log1p(-probability)).mean(),
            'accuracy': ((probability >= 0.5) == y).mean(),
            'mean_probability': probability.mean(),
        }
    )


In [ ]:
covtype_model = build_covtype_model(covtype_X_train, covtype_y_train, covtype_feature_names)

covtype_reference_start = perf_counter()
with covtype_model:
    covtype_reference_idata = pm.sample(
        draws=COVTYPE_DRAWS,
        tune=COVTYPE_TUNE,
        chains=1,
        cores=1,
        target_accept=0.9,
        random_seed=COVTYPE_RANDOM_SEED + 1,
        progressbar=False,
        compute_convergence_checks=False,
    )
covtype_reference_seconds = perf_counter() - covtype_reference_start
covtype_reference_flat = covtype_flat_draws(covtype_reference_idata)


def fit_covtype_shard(shard_id, X_shard, y_shard, labels, seed):
    shard_model = build_covtype_model(X_shard, y_shard, labels)
    shard_start = perf_counter()
    with shard_model:
        shard_idata = pm.sample(
            draws=COVTYPE_SAMPLE_DRAWS,
            tune=COVTYPE_TUNE,
            chains=1,
            cores=1,
            target_accept=0.9,
            random_seed=seed,
            progressbar=False,
            compute_convergence_checks=False,
        )
    return {
        'shard': shard_id,
        'seconds': perf_counter() - shard_start,
        'draws': covtype_flat_draws(shard_idata),
        'n_train': X_shard.shape[0],
    }


covtype_shard_indices = np.array_split(np.arange(covtype_X_train.shape[0]), COVTYPE_NUM_SHARDS)
covtype_parallel_start = perf_counter()
covtype_shard_results = Parallel(n_jobs=COVTYPE_N_JOBS)(
    delayed(fit_covtype_shard)(
        shard_id,
        covtype_X_train[shard_idx],
        covtype_y_train[shard_idx],
        covtype_feature_names,
        COVTYPE_RANDOM_SEED + 10 + shard_id,
    )
    for shard_id, shard_idx in enumerate(covtype_shard_indices)
)
covtype_parallel_seconds = perf_counter() - covtype_parallel_start

covtype_subposteriors = np.stack([result['draws'] for result in covtype_shard_results])
covtype_merge_start = perf_counter()
covtype_consensus_flat = merge_consensus(
    covtype_subposteriors,
    draws=COVTYPE_DRAWS,
    diagonal=False,
    random_seed=COVTYPE_RANDOM_SEED + 100,
)
covtype_merge_seconds = perf_counter() - covtype_merge_start


In [ ]:
covtype_shard_timing = pd.DataFrame(
    [
        {
            'shard': result['shard'],
            'n_train': result['n_train'],
            'seconds': result['seconds'],
        }
        for result in covtype_shard_results
    ]
)
covtype_consensus_seconds = covtype_parallel_seconds + covtype_merge_seconds
covtype_speed_table = pd.DataFrame(
    [
        {
            'fit': 'full_data_nuts',
            'wall_seconds': covtype_reference_seconds,
            'speedup_vs_full': 1.0,
            'n_jobs': 1,
            'merge_seconds': 0.0,
        },
        {
            'fit': 'parallel_consensus_mc',
            'wall_seconds': covtype_consensus_seconds,
            'speedup_vs_full': covtype_reference_seconds / covtype_consensus_seconds,
            'n_jobs': COVTYPE_N_JOBS,
            'merge_seconds': covtype_merge_seconds,
        },
    ]
)
covtype_quality_table = pd.DataFrame(
    {
        'full_data_nuts': covtype_binary_metrics(
            covtype_reference_flat, covtype_X_eval, covtype_y_eval
        ),
        'parallel_consensus_mc': covtype_binary_metrics(
            covtype_consensus_flat, covtype_X_eval, covtype_y_eval
        ),
    }
).T
covtype_shape_checks = pd.Series(
    {
        'reference_flat_shape': covtype_reference_flat.shape,
        'subposteriors_shape': covtype_subposteriors.shape,
        'consensus_flat_shape': covtype_consensus_flat.shape,
    },
    name='shape',
)

print(covtype_shard_timing.to_string(index=False))
print(covtype_speed_table.to_string(index=False))
print(covtype_quality_table.to_string())
print(covtype_shape_checks.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(covtype_speed_table['fit'], covtype_speed_table['wall_seconds'])
ax.set_ylabel('wall seconds')
ax.set_title('Full-data NUTS vs parallel Consensus MC on Forest CoverType')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()


## Standalone merge utilities

The standalone utilities operate on an explicit `subposteriors` array with shape `(num_shards, num_samples, n_params)`. All shards must use the same flattened parameter-column order. Full-covariance consensus and parametric merging also require `num_samples > n_params`; this example uses 300 samples and 9 parameters.

The synthetic array below is pedagogical. It is shaped like four shard posteriors for the same nonlinear basis coefficients, but it is not extracted from the fitted model because `fit_consensus_mc` returns merged posterior draws and metadata, not the intermediate shard draws.


In [ ]:
utility_rng = np.random.default_rng(RANDOM_SEED + 100)
n_params = len(COEFFICIENTS)
centers = TRUE_BETA + utility_rng.normal(scale=0.05, size=(NUM_SHARDS, n_params))
scales = np.linspace(0.05, 0.12, n_params)
toy_subposteriors = utility_rng.normal(
    loc=centers[:, None, :],
    scale=scales[None, None, :],
    size=(NUM_SHARDS, SAMPLE_DRAWS, n_params),
)
toy_consensus_full = merge_consensus(
    toy_subposteriors, draws=400, diagonal=False, random_seed=RANDOM_SEED + 101
)
toy_consensus_diag = merge_consensus(
    toy_subposteriors, draws=400, diagonal=True, random_seed=RANDOM_SEED + 102
)
parametric_mean, parametric_cov = estimate_parametric(toy_subposteriors, diagonal=False)
parametric_diag_mean, parametric_diag_var = estimate_parametric(toy_subposteriors, diagonal=True)
toy_parametric_full = merge_parametric(
    toy_subposteriors, draws=400, diagonal=False, random_seed=RANDOM_SEED + 103
)
toy_parametric_diag = merge_parametric(
    toy_subposteriors, draws=400, diagonal=True, random_seed=RANDOM_SEED + 104
)
toy_consensus_full_df = pd.DataFrame(toy_consensus_full, columns=COEFFICIENTS)
toy_parametric_full_df = pd.DataFrame(toy_parametric_full, columns=COEFFICIENTS)
column_map = pd.Series(COEFFICIENTS, name='parameter_name').rename_axis('column')

expected_draw_shape = (400, n_params)
utility_shapes = pd.Series(
    {
        'toy_consensus_full': toy_consensus_full.shape,
        'toy_consensus_diag': toy_consensus_diag.shape,
        'parametric_mean': parametric_mean.shape,
        'parametric_cov': parametric_cov.shape,
        'parametric_diag_mean': parametric_diag_mean.shape,
        'parametric_diag_var': parametric_diag_var.shape,
        'toy_parametric_full': toy_parametric_full.shape,
        'toy_parametric_diag': toy_parametric_diag.shape,
    },
    name='shape',
)
utility_checks = pd.Series(
    {
        'toy_consensus_full_shape': toy_consensus_full.shape == expected_draw_shape,
        'toy_consensus_diag_shape': toy_consensus_diag.shape == expected_draw_shape,
        'parametric_mean_shape': parametric_mean.shape == (n_params,),
        'parametric_cov_shape': parametric_cov.shape == (n_params, n_params),
        'parametric_diag_mean_shape': parametric_diag_mean.shape == (n_params,),
        'parametric_diag_var_shape': parametric_diag_var.shape == (n_params,),
        'toy_parametric_full_shape': toy_parametric_full.shape == expected_draw_shape,
        'toy_parametric_diag_shape': toy_parametric_diag.shape == expected_draw_shape,
        'toy_consensus_full_columns': list(toy_consensus_full_df.columns) == COEFFICIENTS,
        'toy_parametric_full_columns': list(toy_parametric_full_df.columns) == COEFFICIENTS,
    },
    name='matches_expected',
)

print(utility_shapes.to_string())
print(utility_checks.to_string())
print(column_map.to_string())
print(toy_consensus_full_df.head().to_string(index=False))
print(toy_parametric_full_df.head().to_string(index=False))
